In [0]:

dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("window_end_utc", "")


In [0]:
%run ./common

In [0]:
# Runs independently BEFORE driver for an INCREMENTAL batch.
import os, json, uuid, time, traceback, requests
from pathlib import Path
from datetime import datetime, date, timezone, timedelta
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark.conf.set('spark.sql.session.timeZone', 'UTC')
dbutils.widgets.text('batch_id', '')
dbutils.widgets.text('run_date', '')
dbutils.widgets.text('window_end_utc', '')  # optional explicit YYYY-MM-DDTHH:MM:SSZ
batch_id = dbutils.widgets.get('batch_id').strip()
run_date = dbutils.widgets.get('run_date').strip()
end_text = dbutils.widgets.get('window_end_utc').strip()
if not batch_id or not __import__('re').fullmatch(r'[A-Za-z0-9_-]+', batch_id):
    raise ValueError('Supply safe batch_id (letters, numbers, _ or -).')
date.fromisoformat(run_date)
# Default is end-of-run for a live job; explicit end supports historical backfills.
upper = parse_iso(end_text) if end_text else now()
assert upper <= now() + timedelta(seconds=5), 'window_end_utc cannot be in future'
start = now(); exec_id = str(uuid.uuid4()); rows = 0

try:
    TOKEN = dbutils.secrets.get(scope='github', key='token')
except Exception:
    TOKEN = os.environ.get('GITHUB_TOKEN')
if not TOKEN: raise ValueError('Set secret github/token or GITHUB_TOKEN')

session = requests.Session()
session.headers.update({'Authorization': f'Bearer {TOKEN}',
  'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'})

# Strictly require baseline watermark for each source; never guess from present time.
def read_watermark(repo, entity):
    matches = (spark.table(WM_TABLE)
       .filter((F.col('repository') == repo) & (F.col('entity') == entity) &
               (F.col('layer') == 'RAW'))
       .select('last_successful_watermark').collect())
    if len(matches) != 1 or matches[0][0] is None:
        raise RuntimeError(f'Initialize exactly one RAW watermark for {repo}/{entity}')
    return matches[0][0]

def request(url, params=None):
    for attempt in range(8):
        try:
            r = session.get(url, params=params, timeout=90)
            if r.ok: return r
            if r.status_code in (403, 429):
                if r.headers.get('X-RateLimit-Remaining') == '0':
                    wait = max(int(r.headers.get('X-RateLimit-Reset','0')) - int(time.time()), 1) + 5
                else:
                    wait = int(r.headers.get('Retry-After','60')) if r.headers.get('Retry-After','60').isdigit() else 60
                time.sleep(min(wait, 3600)); continue
            if r.status_code >= 500:
                time.sleep(min(2**attempt,60)); continue
            r.raise_for_status()
        except requests.RequestException:
            if attempt == 7: raise
            time.sleep(min(2**attempt,60))
    raise RuntimeError(f'GitHub request failed after retries: {url}')

def paginate(url, params):
    params = {**params, 'per_page':100}
    while url:
        response = request(url, params)
        records = response.json()
        if not isinstance(records, list):
            raise TypeError('API did not return a JSON list')
        yield from records
        url = response.links.get('next', {}).get('url')
        params = None

# Snapshot checkpoint state up front to avoid an unknown starting baseline.
checkpoints = {(repo, entity):read_watermark(repo,entity)
               for repo in REPOS for entity in ENTITIES}
if any(value >= upper for value in checkpoints.values()):
    raise ValueError('window_end_utc must be later than every selected RAW checkpoint')

manifest_path = Path(ROOT)/'_manifests'/f'incremental_{batch_id}.json'
if manifest_path.exists():
    raise FileExistsError('Batch already published; reuse manifest/driver or choose a new batch_id')

published = []
results = []
try:
    for repo in REPOS:
        for entity in ENTITIES:
            previous = checkpoints[(repo,entity)]
            lower = previous - timedelta(minutes=5)
            endpoint = f'https://api.github.com/repos/{repo}/{entity}'
            params = ({'since':iso(lower),'until':iso(upper)} if entity == 'commits'
                else {'state':'all', 'since':iso(lower), 'sort':'updated', 'direction':'desc'}
                if entity == 'issues' else
                {'state':'all','sort':'updated','direction':'desc'})
            folder = Path(ROOT)/entity/repo.replace('/','__')/f'run_date={run_date}'
            folder.mkdir(parents=True, exist_ok=True)
            target = folder/f'{entity}_incremental_{batch_id}.jsonl'
            part = folder/f'.{entity}_incremental_{batch_id}.part'
            if target.exists():
                raise FileExistsError(f'Refusing to overwrite published batch file: {target}')
            count = 0; num_bytes = 0
            try:
                with open(part, 'w', encoding='utf-8') as out:
                    for item in paginate(endpoint,params):
                        if entity in ('issues','pulls'):
                            updated = item.get('updated_at')
                            if not updated: raise ValueError(f'Missing updated_at from {endpoint}')
                            stamp = parse_iso(updated)
                            if stamp < lower: break
                            if stamp > upper: continue
                        if entity == 'issues' and 'pull_request' in item: continue
                        item = dict(item)
                        item.update({'_repo':repo,'_ingested_at':iso(now()),
                                     '_batch_id':batch_id,'_load_type':'INCREMENTAL',
                                     '_source_endpoint':endpoint,'load_timestamp':iso(now())})
                        line = json.dumps(item,ensure_ascii=False,separators=(',',':'))+'\n'
                        out.write(line)
                        num_bytes += len(line.encode('utf-8')); count += 1
                    out.flush(); os.fsync(out.fileno())
                os.replace(part,target)
                published.append(target)
            except Exception:
                part.unlink(missing_ok=True)
                raise
            rows += count
            results.append({'repository':repo,'entity':entity,'records':count,
                            'bytes':num_bytes,'path':str(target),
                            'window_start':iso(lower),'window_end':iso(upper)})
            print(f'{repo:30} {entity:8} {count:>7,} rows | {num_bytes/1048576:.2f} MiB')

    # Publish manifest BEFORE watermark changes; on failure, do not delete files.
    manifest_path.parent.mkdir(parents=True,exist_ok=True)
    manifest = {'batch_id':batch_id,'run_date':run_date,'load_type':'INCREMENTAL',
                'window_end':iso(upper),'files':results,'total_records':rows}
    part_manifest = manifest_path.with_name('.'+manifest_path.name+'.part')
    part_manifest.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    os.replace(part_manifest,manifest_path)

    # NOTE: 9 MERGEs are independent Delta transactions. No concurrent ingestion runs.
    # A failed update may leave some checkpoints advanced; published files must be processed.
    for repo,entity in checkpoints:
        previous = checkpoints[(repo,entity)]
        # optimistic check (not a transactional cross-table lock)
        if read_watermark(repo,entity) != previous:
            raise RuntimeError(f'Checkpoint changed during ingestion: {repo}/{entity}')
        source = spark.createDataFrame([(repo,entity,'RAW',upper,batch_id,now())], WM_SCHEMA)
        (DeltaTable.forName(spark,WM_TABLE).alias('t')
         .merge(source.alias('s'),"t.repository=s.repository AND t.entity=s.entity AND t.layer=s.layer")
         .whenMatchedUpdate(condition='t.last_successful_watermark < s.last_successful_watermark',
            set={'last_successful_watermark':'s.last_successful_watermark',
                 'batch_id':'s.batch_id','updated_at':'s.updated_at'})
         .execute())
    log(spark,exec_id,batch_id,'incremental_api_ingestion','RAW',run_date,
        'INCREMENTAL',start,'SUCCESS',rows)
    print('PUBLISHED:',manifest_path,'TOTAL ROWS:',rows)
except Exception:
    error = traceback.format_exc()
    print(error)
    # Keep successfully published files for recovery; do not issue another batch blindly.
    try: log(spark,exec_id,batch_id,'incremental_api_ingestion','RAW',run_date,
             'INCREMENTAL',start,'FAILED',rows,error)
    except Exception as log_error: print('AUDIT FAILURE:',log_error)
    raise
finally:
    session.close()
